# 01-3 · Real-restaurant catalogs

| | |
|---|---|
| **What it does** | builds a city's catalog from Uber Eats menus, each dish matched to its closest Food.com recipe; checks how much the restaurants differ in taste; scores the hand-check sheets; checks the loader contract |
| **Proves** | the real catalogs are measured inputs: matcher accuracy and the bitter list are hand-checked, and every catalog passes the contract before the harness may score it |
| **Writes** | only when asked: `data/shared/cache/catalog_ubereats_<city>.json` and `data/shared/validation/match_check_<city>.csv` |
| **Needs** | the Kaggle downloads in `data/shared/external/` to build a catalog (see `data/README.md`); the built catalogs for the other sections |

No API key needed. By default nothing is rebuilt: set `BUILD_CITY` in the first
section to rebuild a catalog.

In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
os.chdir(_root)   # paths below are repo-relative

In [2]:
import nbio
nbio.bootstrap()

repo     : /project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator')

## Step 1 — build a city's catalog

Dish taste is `nbio.recipe_taste(recipe)` of the Food.com recipe the dish is
matched to. Matching is TF-IDF: word 1-2-grams plus character 3-5-grams, fitted
on the Food.com recipe names, cosine top-1. Recipes that share a name are
represented by their medoid (the one whose taste is closest to the group mean).
The dish name is matched first; the description is used only when the name alone
scores below the threshold (0.30). Below the threshold a dish stays on the menu
but unmatched: taste all zeros, ingredients and allergens `None` (unknown, so the
safety filter fails closed).

Restaurants are filtered to food sellers (no grocery, alcohol, pharmacy or
drink-only shops), one branch per chain, non-dish menu items (drinks, sauces,
add-ons, utensils) are dropped, and only restaurants with at least 10 dishes are
kept; `N_RESTAURANTS` of them are sampled with `BUILD_SEED`.

Writes `data/shared/cache/catalog_ubereats_<city>.json` (read by
`nbio.load_catalog_ubereats`) and `data/shared/validation/match_check_<city>.csv`, 100 seeded
dishes for hand-checking the matcher (never overwritten, since it holds labels).
About a minute per city. `LIST_CITIES = True` prints the cities with at least 200
distinct food restaurants. Needs pandas, numpy, scipy and scikit-learn.

In [3]:
# Parameters
LIST_CITIES = False        # True: list the cities in the Uber Eats data
BUILD_CITY = None          # e.g. "Houston, TX" -- 'City, ST' exactly as LIST_CITIES prints it
N_RESTAURANTS = 300        # restaurants to sample (200-500)
BUILD_SEED = 7

In [4]:
import json, random, re, time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

ROOT = nbio.HERE
UBEREATS_DIR = Path(os.environ.get("FM_UBEREATS_DIR") or nbio.EXTERNAL / "ubereats")
VALIDATION = ROOT / "data" / "shared" / "validation"

# Restaurants that do not sell meals. Matched against the comma-separated
# Uber Eats category tags, case-insensitive.
NON_FOOD_TAGS = {"convenience", "grocery", "alcohol", "liquor", "wine & spirits", "pharmacy",
                 "flowers", "pet supplies", "retail", "everyday essentials", "specialty foods"}
DRINK_TAGS = {"coffee and tea", "coffee & tea", "bubble tea", "juice and smoothies",
              "smoothies", "juice", "coffee", "tea"}

# Menu items that are not dishes. A section name that matches drops every item
# in it; an item name that matches the second pattern drops just that item.
NON_DISH_SECTION = re.compile(
    r"\b(drinks?|beverages?|sodas?|bottled|juices?|smoothies?|shakes?|coffee|teas?|cocktails?|"
    r"beers?|wines?|alcohol|liquor|spirits|sauces?|dressings?|dips?|condiments?|extras?|"
    r"add[- ]?ons?|utensils?|napkins?|merch(andise)?|gift cards?)\b", re.I)
NON_DISH_ITEM = re.compile(
    r"^(side of |extra |add |no )|"
    r"\b(coke|coca[- ]cola|pepsi|sprite|dr\.? pepper|fanta|mountain dew|7[- ]?up|gatorade|"
    r"red bull|bottled water|water|soda|lemonade|iced tea|sweet tea|milkshake|smoothie|latte|"
    r"cappuccino|americano|espresso|utensils?|silverware|napkins?|cutlery|straws?)\b", re.I)
SAUCE_ONLY = re.compile(r"^\W*(\w+\W+){0,2}(sauce|dressing|dip|ketchup|ranch|mayo|salsa)\W*$", re.I)

MIN_DISHES = 10
MATCH_THRESHOLD = 0.30
HANDCHECK_N = 100

In [5]:
def parse_city(address: str):
    """'5215 Kirby Dr, Houston, TX, 77098' -> 'Houston, TX'. The state is the
    two-letter part; the city is the part before it."""
    parts = [p.strip() for p in str(address or "").split(",")]
    for i in range(1, len(parts)):
        if re.fullmatch(r"[A-Z]{2}", parts[i]):
            return f"{parts[i - 1]}, {parts[i]}"
    return None


def load_restaurants() -> pd.DataFrame:
    path = UBEREATS_DIR / "restaurants.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found — download the Uber Eats dataset first (data/README.md).")
    df = pd.read_csv(path)
    df["city"] = df["full_address"].map(parse_city)
    return df


def load_menus(restaurant_ids) -> pd.DataFrame:
    path = UBEREATS_DIR / "restaurant-menus.csv"
    ids = set(restaurant_ids)
    chunks = []
    for chunk in pd.read_csv(path, chunksize=500_000):
        chunks.append(chunk[chunk["restaurant_id"].isin(ids)])
    return pd.concat(chunks, ignore_index=True)


def is_food_restaurant(category) -> bool:
    tags = {t.strip().lower() for t in str(category or "").split(",") if t.strip()}
    if tags & NON_FOOD_TAGS:
        return False
    return not (tags and tags <= DRINK_TAGS)


def is_dish(section, name) -> bool:
    name = str(name or "").strip()
    if not name:
        return False
    if NON_DISH_SECTION.search(str(section or "")):
        return False
    return not (NON_DISH_ITEM.search(name) or SAUCE_ONLY.match(name))


def list_cities(min_restaurants: int = 200) -> None:
    rest = load_restaurants()
    rest = rest[rest["category"].map(is_food_restaurant)]
    counts = rest.groupby("city").agg(restaurants=("id", "size"),
                                      unique_names=("name", "nunique"))
    counts = counts[counts["unique_names"] >= min_restaurants].sort_values("unique_names", ascending=False)
    print(f"cities with >= {min_restaurants} distinct food restaurants: {len(counts)}\n")
    for city, row in counts.head(40).iterrows():
        tags = (rest.loc[rest["city"] == city, "category"].fillna("").str.split(",").explode()
                .str.strip().str.lower())
        tags = tags[(tags != "") & ~tags.isin(["american", "fast food", "burgers", "sandwiches"])]
        top = ", ".join(f"{t} {n}" for t, n in tags.value_counts().head(5).items())
        print(f"{city:<28} {row.restaurants:>5} rest  {row.unique_names:>5} distinct   {top}")


def select_restaurants(city: str, n: int, seed: int) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    rest = load_restaurants()
    here = rest[rest["city"].str.lower() == city.lower()]
    if here.empty:
        raise ValueError(f"no restaurants for city {city!r}; set LIST_CITIES = True (format 'City, ST').")
    stats = {"in_city": int(len(here))}
    here = here[here["category"].map(is_food_restaurant)]
    stats["food_restaurants"] = int(len(here))
    # One branch per chain: 20 identical McDonald's menus would count 20 times.
    here = here.sort_values("id").drop_duplicates(subset="name", keep="first")
    stats["after_chain_dedupe"] = int(len(here))

    menus = load_menus(here["id"])
    stats["menu_items"] = int(len(menus))
    keep = [is_dish(s, nm) for s, nm in zip(menus["category"], menus["name"])]
    dishes = menus[keep].copy()
    stats["dropped_non_dishes"] = int(len(menus) - len(dishes))
    dishes = dishes.drop_duplicates(subset=["restaurant_id", "name"])

    per = dishes.groupby("restaurant_id").size()
    eligible = here[here["id"].isin(per[per >= MIN_DISHES].index)]
    stats[f"with_>={MIN_DISHES}_dishes"] = int(len(eligible))
    ids = sorted(eligible["id"].tolist())
    picked = sorted(random.Random(seed).sample(ids, min(n, len(ids))))
    stats["selected"] = len(picked)
    return (eligible[eligible["id"].isin(picked)].sort_values("id"),
            dishes[dishes["restaurant_id"].isin(picked)], stats)

In [6]:
def _norm(s) -> str:
    return re.sub(r"\s+", " ", str(s or "").lower()).strip()


def _medoid(ids):
    """Of several recipes sharing one name, the one whose taste is closest to
    their mean — a typical version, not whichever has the smallest id. Taste,
    ingredients and allergens then all come from that one coherent recipe."""
    if len(ids) == 1:
        return ids[0]
    vecs = np.array([nbio.recipe_taste(i) for i in ids])
    return ids[int(np.argmin(((vecs - vecs.mean(axis=0)) ** 2).sum(axis=1)))]


class RecipeMatcher:
    def __init__(self):
        recipes = nbio._recipes()
        groups = {}
        with open(nbio.flavour_csv_path(), newline="") as f:
            import csv
            csv.field_size_limit(1 << 24)
            for row in csv.DictReader(f):
                name, rid = _norm(row["name"]), int(row["id"])
                if name and rid in recipes:
                    groups.setdefault(name, []).append(rid)
        self.names = sorted(groups)
        self.ids = np.array([_medoid(sorted(groups[n])) for n in self.names])
        self.group_size = np.array([len(groups[n]) for n in self.names])
        self.word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2,
                                    sublinear_tf=True, dtype=np.float32)
        self.char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                                    sublinear_tf=True, dtype=np.float32)
        self.R = self._stack(self.word.fit_transform(self.names),
                             self.char.fit_transform(self.names)).T.tocsr()

    @staticmethod
    def _stack(w, c):
        # Equal weight to word and char evidence; rows re-normalised so the
        # product below is a cosine in [0, 1].
        m = sparse.hstack([w, c]).tocsr()
        norms = np.sqrt(m.multiply(m).sum(axis=1)).A1
        norms[norms == 0] = 1.0
        return sparse.diags(1.0 / norms).dot(m).astype(np.float32)

    def top_k(self, texts, k=3, batch=256):
        Q = self._stack(self.word.transform(texts), self.char.transform(texts))
        idx = np.zeros((len(texts), k), dtype=np.int64)
        val = np.zeros((len(texts), k), dtype=np.float32)
        for s in range(0, len(texts), batch):
            S = Q[s:s + batch].dot(self.R).toarray()
            part = np.argpartition(-S, k, axis=1)[:, :k]
            ps = np.take_along_axis(S, part, axis=1)
            order = np.argsort(-ps, axis=1, kind="stable")
            idx[s:s + batch] = np.take_along_axis(part, order, axis=1)
            val[s:s + batch] = np.take_along_axis(ps, order, axis=1)
        return idx, val


def match_dishes(dishes: pd.DataFrame) -> pd.DataFrame:
    t0 = time.time()
    m = RecipeMatcher()
    print(f"  TF-IDF fitted on {len(m.names):,} recipe names ({time.time() - t0:.0f}s)")

    names = dishes["name"].map(_norm)
    descs = dishes["description"].fillna("").map(_norm)
    uniq = sorted(set(names))
    ui, uv = m.top_k(uniq)
    by_name = {t: (ui[j], uv[j]) for j, t in enumerate(uniq)}

    # Fallback: name + description, only for names that miss the threshold.
    weak = sorted({f"{n} {d}".strip() for n, d in zip(names, descs)
                   if by_name[n][1][0] < MATCH_THRESHOLD and d})
    by_full = {}
    if weak:
        wi, wv = m.top_k(weak)
        by_full = {t: (wi[j], wv[j]) for j, t in enumerate(weak)}
    print(f"  matched {len(uniq):,} distinct dish names, {len(weak):,} description fallbacks "
          f"({time.time() - t0:.0f}s)")

    out = []
    for n, d in zip(names, descs):
        idx, val, used = *by_name[n], "name"
        full = f"{n} {d}".strip()
        if full in by_full and by_full[full][1][0] > val[0]:
            idx, val = by_full[full]
            used = "name+description"
        out.append({"recipe_id": int(m.ids[idx[0]]), "recipe_name": m.names[idx[0]], "same_name_recipes": int(m.group_size[idx[0]]),
                    "match_score": round(float(val[0]), 4), "match_text": used,
                    "alternatives": [m.names[j] for j in idx[1:]]})
    return pd.DataFrame(out, index=dishes.index)

In [7]:
def _price(p):
    try:
        return float(str(p).split()[0])
    except (ValueError, IndexError):
        return None


def build_catalog(city: str, n: int, seed: int) -> None:
    t0 = time.time()
    print(f"[1/4] selecting restaurants in {city} ...")
    rest, dishes, stats = select_restaurants(city, n, seed)
    for k, v in stats.items():
        print(f"      {k:<22} {v:,}")

    print(f"[2/4] matching {len(dishes):,} dishes to Food.com recipes ...")
    dishes = dishes.join(match_dishes(dishes))
    dishes["matched"] = dishes["match_score"] >= MATCH_THRESHOLD

    print("[3/4] dish taste + allergens ...")
    restaurants = []
    for _, r in rest.iterrows():
        menu = []
        for _, d in dishes[dishes["restaurant_id"] == r["id"]].iterrows():
            ings = nbio.recipe_ingredients(d["recipe_id"]) if d["matched"] else None
            menu.append({
                "name": str(d["name"]).strip(),
                "description": None if pd.isna(d["description"]) else str(d["description"]),
                "section": None if pd.isna(d["category"]) else str(d["category"]),
                "price": _price(d["price"]),
                "recipe_id": int(d["recipe_id"]) if d["matched"] else None,
                "recipe_name": d["recipe_name"] if d["matched"] else None,
                "match_score": d["match_score"],
                "match_text": d["match_text"],
                "same_name_recipes": int(d["same_name_recipes"]),
                "taste": nbio.recipe_taste(d["recipe_id"]) if d["matched"] else [0.0] * 6,
                "ingredients": ings,
                "allergens": nbio.dish_allergens(ings),
            })
        restaurants.append({
            "option_id": f"ue_{int(r['id'])}", "name": str(r["name"]),
            "category": None if pd.isna(r["category"]) else str(r["category"]),
            "price_range": None if pd.isna(r.get("price_range")) else str(r["price_range"]),
            "menu": menu,
        })

    stats["dishes"] = int(len(dishes))
    stats["matched_dishes"] = int(dishes["matched"].sum())
    meta = {"source": "kaggle ahmedshahriarsakib/uber-eats-usa-restaurants-menus",
            "city": city, "n_requested": n, "seed": seed, "match_threshold": MATCH_THRESHOLD,
            "min_dishes": MIN_DISHES, "bitter_list_sha": nbio.bitter_list_sha(),
            "built": time.strftime("%Y-%m-%d %H:%M"), "stats": stats}
    out = nbio.ubereats_cache_path(city)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps({"meta": meta, "restaurants": restaurants}, indent=1) + "\n")
    print(f"      -> {out.relative_to(ROOT)} ({out.stat().st_size / 1e6:.1f} MB)")

    print("[4/4] hand-check sheet + summary ...")
    write_handcheck(city, dishes, rest, seed)
    catalog = nbio.load_catalog_ubereats(city)
    sp = nbio.taste_spread(catalog)
    print(f"\n  restaurants        {len(catalog)}")
    print(f"  dishes             {stats['dishes']:,}  (matched {stats['matched_dishes']:,})")
    print(f"  taste coverage     {nbio.taste_coverage(catalog):.1%}  (contract: >= 90%)")
    print(f"  spread (mean std)  {sp['spread']:.4f}   per dim: "
          + "  ".join(f"{d} {v:.3f}" for d, v in sp["per_dim_std"].items()))
    print(f"  done in {time.time() - t0:.0f}s")


def write_handcheck(city, dishes, rest, seed) -> None:
    """100 seeded dishes drawn from ALL dishes (matched or not), so the labels
    measure accuracy and also let the threshold be tuned afterwards."""
    names = dict(zip(rest["id"], rest["name"]))
    sample = dishes.sample(n=min(HANDCHECK_N, len(dishes)), random_state=seed)
    sheet = pd.DataFrame({
        "restaurant": sample["restaurant_id"].map(names),
        "dish": sample["name"],
        "description": sample["description"].fillna(""),
        "matched_recipe": sample["recipe_name"],
        "match_score": sample["match_score"],
        "match_text": sample["match_text"],
        "alt_2": sample["alternatives"].map(lambda a: a[0] if len(a) > 0 else ""),
        "alt_3": sample["alternatives"].map(lambda a: a[1] if len(a) > 1 else ""),
        "correct": "",   # fill in: y = same dish, p = close (same family), n = wrong
        "notes": "",
    })
    VALIDATION.mkdir(exist_ok=True)
    path = VALIDATION / f"match_check_{nbio.city_slug(city)}.csv"
    if path.exists():
        print(f"      {path.relative_to(ROOT)} exists — not overwritten (labels may be in it)")
        return
    sheet.to_csv(path, index=False)
    print(f"      -> {path.relative_to(ROOT)} ({len(sheet)} dishes to hand-label)")

In [8]:
if LIST_CITIES:
    list_cities()
if BUILD_CITY:
    build_catalog(BUILD_CITY, N_RESTAURANTS, BUILD_SEED)
if not (LIST_CITIES or BUILD_CITY):
    built = sorted(p.name for p in nbio.EXTERNAL_CACHE.glob("catalog_ubereats_*.json"))
    print("nothing to build (BUILD_CITY is None). Catalogs on disk:", ", ".join(built) or "none")

nothing to build (BUILD_CITY is None). Catalogs on disk: catalog_ubereats_houston-tx.json, catalog_ubereats_seattle-wa.json


## Step 2 — how much do the restaurants differ in taste?

The spread is the mean, over the 6 tastes, of the standard deviation of the
restaurant taste vectors. Also printed: how far apart restaurants sit (the range
per taste), taste coverage, and the pairwise cosine between restaurants, since
cosine is what satisfaction actually uses (Equation 2). Reported for the small
catalog and for every built Uber Eats city (or only `SPREAD_CITIES`). Nothing is
written.

In [9]:
from itertools import combinations

SPREAD_CITIES = None     # None: every built city; or a list such as ["Houston, TX"]


def report_spread(label: str, catalog) -> dict:
    sp = nbio.taste_spread(catalog)
    vecs = [nbio.resolve_taste_vector(o["taste"]) for o in catalog]
    ranges = {d: max(v[i] for v in vecs) - min(v[i] for v in vecs) for i, d in enumerate(nbio.TASTE_DIMS)}
    cos = [nbio.calculate_cosine_similarity(a, b) for a, b in combinations(vecs, 2)]
    cos_min = min(cos) if cos else 1.0
    cos_mean = sum(cos) / len(cos) if cos else 1.0
    print(f"\n{label}  ({sp['n_restaurants']} restaurants, taste coverage {nbio.taste_coverage(catalog):.1%})")
    print(f"  spread (mean per-dim std)   {sp['spread']:.4f}")
    print("  per-dim std    " + "  ".join(f"{d} {v:.3f}" for d, v in sp["per_dim_std"].items()))
    print("  per-dim range  " + "  ".join(f"{d} {v:.3f}" for d, v in ranges.items()))
    print(f"  restaurant-vs-restaurant cosine: mean {cos_mean:.3f}, min {cos_min:.3f}  "
          f"(1.0 = identical taste direction)")
    return sp


base_spread = report_spread("small catalog (5 restaurants)", nbio._load_catalog_small())
if SPREAD_CITIES:
    spread_cities = SPREAD_CITIES
else:
    spread_cities = [json.loads(p.read_text())["meta"]["city"]
                     for p in sorted(nbio.EXTERNAL_CACHE.glob("catalog_ubereats_*.json"))]
if not spread_cities:
    print("\nno Uber Eats catalog built yet — build one in Step 1")
for c in spread_cities:
    sp = report_spread(f"ubereats: {c}", nbio.load_catalog_ubereats(c))
    ratio = sp["spread"] / base_spread["spread"] if base_spread["spread"] else float("inf")
    print(f"  -> {ratio:.1f}x the small catalog's spread")


small catalog (5 restaurants)  (5 restaurants, taste coverage 46.5%)
  spread (mean per-dim std)   0.0173
  per-dim std    sweet 0.031  salty 0.017  sour 0.022  bitter 0.000  umami 0.027  spicy 0.007
  per-dim range  sweet 0.083  salty 0.050  sour 0.060  bitter 0.000  umami 0.078  spicy 0.021
  restaurant-vs-restaurant cosine: mean 0.981, min 0.956  (1.0 = identical taste direction)



ubereats: Houston, TX  (300 restaurants, taste coverage 95.5%)
  spread (mean per-dim std)   0.0401
  per-dim std    sweet 0.068  salty 0.048  sour 0.026  bitter 0.019  umami 0.051  spicy 0.028
  per-dim range  sweet 0.477  salty 0.290  sour 0.198  bitter 0.172  umami 0.299  spicy 0.162
  restaurant-vs-restaurant cosine: mean 0.948, min 0.441  (1.0 = identical taste direction)
  -> 2.3x the small catalog's spread



ubereats: Seattle, WA  (300 restaurants, taste coverage 95.2%)
  spread (mean per-dim std)   0.0444
  per-dim std    sweet 0.072  salty 0.055  sour 0.026  bitter 0.024  umami 0.060  spicy 0.030
  per-dim range  sweet 0.670  salty 0.380  sour 0.172  bitter 0.155  umami 0.308  spicy 0.148
  restaurant-vs-restaurant cosine: mean 0.940, min 0.331  (1.0 = identical taste direction)
  -> 2.6x the small catalog's spread


## Step 3 — the hand-check sheets and their scores

The two measured components of the food catalog: a spot-check of 100 Food.com
recipes for the hand-made bitter list (50 it scores bitter, 50 it scores 0,
shuffled) and a hand-check of 100 dish-to-recipe matches per city (written by
Step 1). Each sheet is labelled by hand; this step prints the accuracy of each
with 95% Wilson intervals, and the accuracy at other cut-offs. The results are
recorded in `validation/README.md`.

The sheets quote Kaggle text, so they are gitignored local files
(`data/shared/validation/*.csv`); a sheet that is missing is skipped with a message.
`MAKE_BITTER_SHEET = True` writes a new, unlabelled bitter sheet (it never
overwrites an existing one).

In [10]:
import csv

MAKE_BITTER_SHEET = False
BITTER_SHEET = VALIDATION / "bitter_spotcheck.csv"
BITTER_I = nbio.TASTE_DIMS.index("bitter")


def _wilson(k: int, n: int, z: float = 1.96):
    """95% Wilson interval for k successes in n — sane at small n and near 0/1."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / d
    return (max(0.0, c - h), min(1.0, c + h))


def _fmt(k, n):
    lo, hi = _wilson(k, n)
    return f"{k}/{n} = {k / n:.0%}  (95% CI {lo:.0%}-{hi:.0%})" if n else "n/a"


def bitter_sheet(seed: int = 7, n: int = 100) -> None:
    """50 recipes the list scores bitter, 50 it scores not bitter, seeded.
    Stratified so both error types get measured: false bitter (precision)
    and missed bitter (false-negative rate)."""
    if BITTER_SHEET.exists():
        print(f"{BITTER_SHEET.relative_to(ROOT)} exists — not overwriting labels.")
        return
    names = {}
    with open(nbio.flavour_csv_path(), newline="") as f:
        csv.field_size_limit(1 << 24)
        for row in csv.DictReader(f):
            names[int(row["id"])] = row["name"]
    ids = sorted(nbio._recipes())
    pos = [i for i in ids if nbio.recipe_taste(i)[BITTER_I] > 0]
    neg = [i for i in ids if nbio.recipe_taste(i)[BITTER_I] == 0]
    rng = random.Random(seed)
    picked = rng.sample(pos, n // 2) + rng.sample(neg, n - n // 2)
    rng.shuffle(picked)                       # labeller should not see the stratum order
    VALIDATION.mkdir(exist_ok=True)
    with open(BITTER_SHEET, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["recipe_id", "recipe_name", "ingredients", "bitter_score",
                    "bitter_ingredients", "looks_bitter", "notes"])
        for i in picked:
            ings = nbio.recipe_ingredients(i)
            hits = [g for g in ings if nbio.ingredient_bitter(g)]
            w.writerow([i, names.get(i, ""), "; ".join(ings), nbio.recipe_taste(i)[BITTER_I],
                        "; ".join(hits), "", ""])
    print(f"-> {BITTER_SHEET.relative_to(ROOT)}  ({len(pos):,} of {len(ids):,} recipes score bitter > 0; "
          f"sheet has {n // 2} of each)")
    print("   fill looks_bitter with y (a noticeably bitter dish) or n, then run score_bitter()")


def score_bitter() -> None:
    if not BITTER_SHEET.exists():
        print(f"{BITTER_SHEET.relative_to(ROOT)} not found (a gitignored local file) — skipped.")
        return
    rows = list(csv.DictReader(open(BITTER_SHEET)))
    lab = [r for r in rows if r["looks_bitter"].strip().lower() in ("y", "n")]
    if not lab:
        print("no rows labelled yet (looks_bitter = y/n).")
        return
    pos = [r for r in lab if float(r["bitter_score"]) > 0]
    neg = [r for r in lab if float(r["bitter_score"]) == 0]
    tp = sum(r["looks_bitter"].strip().lower() == "y" for r in pos)
    fn = sum(r["looks_bitter"].strip().lower() == "y" for r in neg)
    print(f"labelled {len(lab)}/{len(rows)}")
    print(f"  precision   (scored bitter AND looks bitter)   {_fmt(tp, len(pos))}")
    print(f"  missed rate (scored 0 BUT looks bitter)        {_fmt(fn, len(neg))}")
    # The score is graded (a pinch of zest gives 0.03), so "> 0" is the harshest
    # cut. Precision/recall at higher cuts show whether the size of the score tracks taste.
    print("  cut-off  scored>=cut  precision                      recall")
    n_y = sum(r["looks_bitter"].strip().lower() == "y" for r in lab)
    for t in (0.05, 0.1, 0.2, 0.3):
        hit = [r for r in lab if float(r["bitter_score"]) >= t]
        k = sum(r["looks_bitter"].strip().lower() == "y" for r in hit)
        print(f"    {t:.2f}   {len(hit):>4}       {_fmt(k, len(hit)):<30} {_fmt(k, n_y)}")


def score_match(city: str) -> None:
    path = VALIDATION / f"match_check_{nbio.city_slug(city)}.csv"
    if not path.exists():
        print(f"{path.relative_to(ROOT)} not found (a gitignored local file) — skipped.")
        return
    rows = list(csv.DictReader(open(path)))
    lab = [r for r in rows if r["correct"].strip().lower() in ("y", "p", "n")]
    if not lab:
        print(f"no rows labelled yet in {path.name} (correct = y/p/n).")
        return
    print(f"{city}: labelled {len(lab)}/{len(rows)}")
    y = sum(r["correct"].strip().lower() == "y" for r in lab)
    yp = sum(r["correct"].strip().lower() in ("y", "p") for r in lab)
    print(f"  exact dish          {_fmt(y, len(lab))}")
    print(f"  exact or close      {_fmt(yp, len(lab))}")
    # Accuracy above each threshold: what raising the cut-off would buy.
    print("  threshold  kept  exact-or-close among kept")
    for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
        kept = [r for r in lab if float(r["match_score"]) >= t]
        k = sum(r["correct"].strip().lower() in ("y", "p") for r in kept)
        print(f"    {t:.1f}     {len(kept):>4}  {_fmt(k, len(kept))}")

In [11]:
if MAKE_BITTER_SHEET:
    bitter_sheet()
score_bitter()
print()
for c in ("Houston, TX", "Seattle, WA"):
    score_match(c)
    print()

no rows labelled yet (looks_bitter = y/n).

no rows labelled yet in match_check_houston-tx.csv (correct = y/p/n).

no rows labelled yet in match_check_seattle-wa.csv (correct = y/p/n).



## Step 4 — the loader contract

What every catalog must satisfy before the harness is allowed to score it: a
non-empty catalog with unique option ids; every restaurant and dish has a name
and a taste vector of 6 finite numbers; every dish has an `allergens` field with
known allergens only, and a dish whose ingredients are unknown has unknown
allergens (fail closed); taste coverage is at least 90%. Also checked:
`recipe_taste` itself, the bitter list, the allergen tags, and the shape of the
small catalog (which predates the contract, so it has no coverage floor).

Every Uber Eats catalog built in `data/shared/cache/` is checked (or only
`FM_CITY` when set). A check that needs Kaggle data not on this machine is
skipped with a message, not failed.

In [12]:
import math

MIN_COVERAGE = 0.90


class Skip(Exception):
    pass


def _finite6(vec) -> bool:
    return (isinstance(vec, (list, tuple)) and len(vec) == 6
            and all(isinstance(x, (int, float)) and math.isfinite(x) for x in vec))


def check_catalog(catalog, label: str) -> None:
    assert catalog, f"{label}: catalog is empty"
    ids = [o["option_id"] for o in catalog]
    assert len(ids) == len(set(ids)), f"{label}: duplicate option_id"
    for o in catalog:
        where = f"{label}/{o['option_id']}"
        assert o.get("name"), f"{where}: no name"
        assert _finite6(o["taste"]), f"{where}: restaurant taste is not 6 finite numbers: {o['taste']}"
        assert o.get("menu"), f"{where}: menu is empty"
        for d in o["menu"]:
            assert d.get("name"), f"{where}: dish with no name"
            assert _finite6(d["taste"]), f"{where}/{d.get('name')}: dish taste {d.get('taste')}"
            assert "allergens" in d, f"{where}/{d['name']}: no allergens field"
            a = d["allergens"]
            assert a is None or set(a) <= set(nbio.FDA_ALLERGEN_KEYWORDS), \
                f"{where}/{d['name']}: unknown allergen in {a}"
            # unknown ingredients must stay unknown allergens (fail closed)
            if d.get("ingredients") is None:
                assert a is None, f"{where}/{d['name']}: allergens without ingredients"
    cov = nbio.taste_coverage(catalog)
    assert cov >= MIN_COVERAGE, f"{label}: taste coverage {cov:.1%} < {MIN_COVERAGE:.0%}"


def check_recipe_taste():
    try:
        nbio._recipes()
    except FileNotFoundError as e:
        raise Skip(str(e).splitlines()[0])
    rid = next(iter(nbio._recipes()))
    v = nbio.recipe_taste(rid)
    assert _finite6(v), v
    assert all(0.0 <= x <= 1.0 for x in v), v
    assert nbio.recipe_taste(-1) == [0.0] * 6
    assert nbio.recipe_taste("not an id") == [0.0] * 6
    assert nbio.recipe_ingredients(-1) is None


def check_bitter_list():
    assert nbio.ingredient_bitter("strong brewed coffee") == 0.9
    assert nbio.ingredient_bitter("coffee creamer") == 0.0      # exclusion beats 'coffee'
    assert nbio.ingredient_bitter("white chocolate chips") == 0.0
    assert nbio.ingredient_bitter("sugar") is None


def check_allergen_tags():
    assert nbio.dish_allergens(None) is None
    assert nbio.dish_allergens([]) == []
    assert nbio.dish_allergens(["coconut milk", "eggplant", "water chestnuts"]) == []
    assert nbio.dish_allergens(["soy sauce"]) == ["soy", "wheat"]
    assert nbio.dish_allergens(["peanut butter", "shrimp", "eggs"]) == ["egg", "peanuts", "shellfish"]


def check_ubereats_catalogs():
    city = os.environ.get("FM_CITY")
    paths = ([nbio.ubereats_cache_path(city)] if city
             else sorted(nbio.EXTERNAL_CACHE.glob("catalog_ubereats_*.json")))
    paths = [p for p in paths if p.exists()]
    if not paths:
        raise Skip("no Uber Eats catalog built yet (Step 1)")
    for p in paths:
        c = json.loads(p.read_text())["meta"]["city"]
        check_catalog(nbio.load_catalog_ubereats(c), c)


def check_default_catalog_shape():
    """The small catalog predates the contract: shape only, no coverage
    floor (it is 47%)."""
    for o in nbio._load_catalog_small():
        assert _finite6(o["taste"]) and o["menu"]
        assert all(_finite6(d["taste"]) for d in o["menu"])

In [13]:
failed = 0
for fn in (check_recipe_taste, check_bitter_list, check_allergen_tags,
           check_ubereats_catalogs, check_default_catalog_shape):
    try:
        fn()
        print(f"PASS  {fn.__name__}")
    except Skip as s:
        print(f"SKIP  {fn.__name__}: {s}")
    except AssertionError as e:
        failed += 1
        print(f"FAIL  {fn.__name__}: {e}")
assert failed == 0, f"{failed} contract check(s) failed"

PASS  check_recipe_taste
PASS  check_bitter_list
PASS  check_allergen_tags


PASS  check_ubereats_catalogs
PASS  check_default_catalog_shape
